## 1. Install Dependencies

In [1]:
!pip install hf_transfer
!pip uninstall transformers -y
!pip install transformers==4.48.0
!pip uninstall numpy -y
!pip install 'numpy>=1.26,<2.0'  # Stay on 1.x — unbabel-comet requires <2.0
!pip install sacrebleu>=2.3 unbabel-comet>=2.0.0 jiwer>=3.0.0
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1
!pip install peft>=0.6.0 bitsandbytes>=0.41.0
!pip install indictranstoolkit
!pip install scikit-learn
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 14.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 26.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 47.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]
Found existing installation: numpy 2.1.2
Uninstalling numpy-2.1.2:
  Successfully uninstalled numpy-2.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 39.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 46.2 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 72.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.7 MB/s  0:00:00
   ━━━━━━━

## 2. Hugging Face Authentication
### Regenerate it at https://huggingface.co/settings/tokens and paste it.

In [2]:
import os
from huggingface_hub import login

#  FIX: Use environment variable instead of hardcoding token
# Set this in your terminal before launching Jupyter:
#   export HF_TOKEN="hf_your_new_token_here"
# OR set it directly in a cell (but don't commit the notebook after):
#   os.environ["HF_TOKEN"] = "hf_your_new_token_here"

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print(" Logged in via environment variable")
else:
    print("HF_TOKEN not set — launching interactive login...")
    login()  # Will prompt interactively

HF_TOKEN not set — launching interactive login...


## 3. Verify Package Versions

In [4]:
import numpy
import torch
import transformers

print(f" Numpy:        {numpy.__version__}")
print(f" PyTorch:      {torch.__version__}")
print(f" Transformers: {transformers.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")

 Numpy:        1.26.4
 PyTorch:      2.4.1+cu121
 Transformers: 4.48.0
 CUDA available: True
 GPU: NVIDIA GeForce RTX 3090


## 4. Quick Baseline Test (IndicTrans2 without fine-tuning)

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"

print("Loading tokenizer and model for baseline test...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device)

ip = IndicProcessor(inference=True)

# Quick test on a few sample sentences
test_sentences = [
    "ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ",
    "ಅವನು ತನ್ನ ತಾಯಿಯೊಂದಿಗೆ ಮಾತನಾಡುತ್ತಿದ್ದಾನೆ",
]
src_lang, tgt_lang = "kan_Knda", "kan_Knda"

batch = ip.preprocess_batch(test_sentences, src_lang=src_lang, tgt_lang=tgt_lang)
batch = tokenizer(batch, padding="longest", truncation=True, max_length=256, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model.generate(**batch, num_beams=5, num_return_sequences=1, max_length=256)

decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=True)
decoded = ip.postprocess_batch(decoded, lang=tgt_lang)

print("\n=== BASELINE OUTPUT (before fine-tuning) ===")
for src, out in zip(test_sentences, decoded):
    print(f"  Input:  {src}")
    print(f"  Output: {out}")
    print()

del model  # Free GPU memory before training
torch.cuda.empty_cache()
print(" Baseline done, model unloaded to free GPU memory")

Loading tokenizer and model for baseline test...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


=== BASELINE OUTPUT (before fine-tuning) ===
  Input:  ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ
  Output: ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ.

  Input:  ಅವನು ತನ್ನ ತಾಯಿಯೊಂದಿಗೆ ಮಾತನಾಡುತ್ತಿದ್ದಾನೆ
  Output: ಅವನು ತನ್ನ ತಾಯಿಯೊಂದಿಗೆ ಮಾತನಾಡುತ್ತಿದ್ದಾನೆ

 Baseline done, model unloaded to free GPU memory


## 5. Data Preparation
### Dataset observations from Bookish_Dharwad.json:
- Total: 4000 raw lines
- Mix of conversational, news, literary, and instructional sentences
- Good dialectal variety: verb endings, pronouns, vocabulary all differ consistently

In [6]:
import json
import os
import random
from sklearn.model_selection import train_test_split

random.seed(42)

# Load data
data = []
with open("Bookish_Dharwad.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f"Raw examples loaded: {len(data)}")

#  FIX: Remove duplicates (5 identical rows found at end of dataset)
seen = set()
deduped = []
for item in data:
    key = (item["src"], item["tgt"])
    if key not in seen:
        seen.add(key)
        deduped.append(item)

print(f"After deduplication: {len(deduped)} (removed {len(data) - len(deduped)} duplicates)")
data = deduped

# Shuffle
random.shuffle(data)


train_data, temp_data = train_test_split(data, test_size=0.10, random_state=42)
dev_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"\nSplit sizes:")
print(f"  Train: {len(train_data)}")
print(f"  Dev:   {len(dev_data)}")
print(f"  Test:  {len(test_data)}")

# Save splits
for name, split_data in [("train", train_data), ("dev", dev_data), ("test", test_data)]:
    with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
        for item in split_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f" Saved {name}.jsonl")

Raw examples loaded: 4005
After deduplication: 4000 (removed 5 duplicates)

Split sizes:
  Train: 3600
  Dev:   200
  Test:  200
 Saved train.jsonl
 Saved dev.jsonl
 Saved test.jsonl


## 6. Inspect LoRA Target Modules
Run this cell to find the actual attention module names in IndicTrans2 before setting `target_modules`.

In [7]:
# NEW: Inspect actual module names to set correct LoRA target_modules
from transformers import AutoModelForSeq2SeqLM
import torch

temp_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
)

print("=== Attention-related modules in IndicTrans2 ===")
attn_modules = set()
for name, module in temp_model.named_modules():
    if any(kw in name.lower() for kw in ["attn", "attention", "q_proj", "v_proj", "k_proj", "out_proj"]):
        leaf_name = name.split(".")[-1]
        attn_modules.add(leaf_name)

print("Unique leaf module names:", sorted(attn_modules))
print("\nThese are the valid values for target_modules in LoraConfig.")

del temp_model
torch.cuda.empty_cache()

=== Attention-related modules in IndicTrans2 ===
Unique leaf module names: ['encoder_attn', 'encoder_attn_layer_norm', 'k_proj', 'out_proj', 'q_proj', 'self_attn', 'self_attn_layer_norm', 'v_proj']

These are the valid values for target_modules in LoraConfig.


## 7. Training Setup

In [ ]:
import torch
import json
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,   
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
from IndicTransToolkit import IndicProcessor

MODEL_NAME = "ai4bharat/indictrans2-indic-indic-1B"
OUTPUT_DIR = "./dharwad-lora-model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# LoRA hyperparameters
LORA_R = 1024           
                       # more adapter capacity to push loss further.
LORA_ALPHA = 2048       # 2 * LORA_R (keeping the ratio)
LORA_DROPOUT = 0.05    #  Reduced from 0.1 — not needed given the tight train/eval gap

# Training hyperparameters
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 8   # Effective batch = 8 * 8 = 64
LEARNING_RATE = 5e-5   #  FIX: Lower LR (was 2e-4) — safer for small dataset
EPOCHS = 20
MAX_LENGTH = 128

print(f"Using device: {DEVICE}")

# ============================================
# LOAD DATA
# ============================================
def load_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

train_data = load_jsonl("train.jsonl")
dev_data = load_jsonl("dev.jsonl")

print(f"Train examples: {len(train_data)}")
print(f"Dev examples:   {len(dev_data)}")

train_dataset = Dataset.from_list(train_data)
dev_dataset = Dataset.from_list(dev_data)

# ============================================
# LOAD MODEL AND TOKENIZER
# ============================================
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE)

print(" Model loaded")

Using device: cuda
Train examples: 3600
Dev examples:   200

Loading tokenizer...
Loading model...
 Model loaded


## 8. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# For IndicTrans2 (fairseq-based), the correct names are typically:
# "q_proj" and "v_proj"  — targeting only these is safer for small data.
# If the inspector showed different names, update this list.
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

print("Applying LoRA configuration...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES, 
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

trainable_params = model.num_parameters(only_trainable=True)
total_params = model.num_parameters()
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters:     {total_params:,}")
print("\n(A low trainable % is good — prevents overfitting on small data)")

Applying LoRA configuration...
Trainable parameters: 226,492,416 (15.79%)
Total parameters:     1,434,589,184

(A low trainable % is good — prevents overfitting on small data)


## 9. Tokenization
### Key fixes: `text_target=` instead of deprecated `as_target_tokenizer()`, and padding labels set to -100

In [ ]:
# ============================================
# TOKENIZATION FUNCTION
# ============================================
ip = IndicProcessor(inference=False)

SRC_LANG = "kan_Knda"
TGT_LANG = "kan_Knda"


def _strip_tag_prefix(text: str, src_lang: str, tgt_lang: str) -> str:
    """Remove the 'src_lang tgt_lang ' prefix IndicProcessor adds, since the
    TARGET tokenizer (_tgt_tokenize) must receive plain text only."""
    prefix = f"{src_lang} {tgt_lang} "
    if text.startswith(prefix):
        return text[len(prefix):]
    return text  # already clean / prefix not present

def preprocess_function(examples):
    # Source side: IndicProcessor's tag prefix is REQUIRED here — _src_tokenize
    # parses it explicitly (text.split(" ", 2)) and validates against LANGUAGE_TAGS.
    src_texts = ip.preprocess_batch(
        examples["src"],
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG
    )

    inputs = tokenizer(
        src_texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )

    # Target side: run IndicProcessor for its script normalization benefits,
    # then STRIP the tag prefix it adds — _tgt_tokenize takes plain text only.
    tgt_texts_raw = ip.preprocess_batch(
        examples["tgt"],
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG
    )
    tgt_texts = [_strip_tag_prefix(t, SRC_LANG, TGT_LANG) for t in tgt_texts_raw]

    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = TGT_LANG

    labels = tokenizer(
        text_target=tgt_texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors=None
    )

    label_ids = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]

    inputs["labels"] = label_ids
    return inputs

# --- Sanity check before tokenizing the full dataset ---
_test = preprocess_function({"src": [train_data[0]["src"]], "tgt": [train_data[0]["tgt"]]})
_label_ids_clean = [t for t in _test["labels"][0] if t != -100]
_label_tokens = tokenizer.convert_ids_to_tokens(_label_ids_clean[:8])
print("Sanity check — first 8 LABEL tokens (should be real Kannada/Devanagari")
print("word-pieces, NOT 'kan'/'_'/'K'/'nd'/'a' style fragments):")
print(" ", _label_tokens)

_decoded_prefix = tokenizer.convert_tokens_to_string(_label_tokens[:5])
_tag_leak_signature = ("kan" in _decoded_prefix.replace(" ", "").lower()
                        and "knda" in _decoded_prefix.replace(" ", "").lower())
assert not _tag_leak_signature, (
    "Tag text appears to be leaking into labels again — check _strip_tag_prefix."
)
print(" Looks clean — no tag-text leakage detected at the start of labels.")


# Tokenize datasets
print("Tokenizing datasets with IndicProcessor...")
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)
tokenized_dev = dev_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dev_dataset.column_names,
    desc="Tokenizing dev"
)

print(f" Train: {len(tokenized_train)} examples")
print(f" Dev:   {len(tokenized_dev)} examples")

# Sanity check: print a decoded sample to verify it looks like Kannada
sample_ids = [t for t in tokenized_train[0]["input_ids"] if t != tokenizer.pad_token_id]
decoded_sample = tokenizer.decode(sample_ids, skip_special_tokens=False)
print(f"\nSample input (decoded back): {decoded_sample[:100]}")

Parameter 'function'=<function preprocess_function at 0x7f7bac188680> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Sanity check — first 8 LABEL tokens (should be real Kannada/Devanagari
word-pieces, NOT 'kan'/'_'/'K'/'nd'/'a' style fragments):
  ['▁ट्रिल', '▁.', '▁हिट्', '▁प्रभ', 'णमॆन्नावश्य', '▁मळ्ळी', '▁इदु', '▁सभ्युल']
 Looks clean — no tag-text leakage detected at the start of labels.
Tokenizing datasets with IndicProcessor...


Tokenizing train:   0%|          | 0/3600 [00:00<?, ? examples/s]

Tokenizing dev:   0%|          | 0/200 [00:00<?, ? examples/s]

 Train: 3600 examples
 Dev:   200 examples

Sample input (decoded back): ൽൽातही ।श्बासच बिघड चॆब था सुपारिश किया कोसं चॆब बंधन,</s>


## 10. Training Arguments

In [ ]:
# ============================================
# TRAINING ARGUMENTS
# ============================================


training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    #  FIX: Epoch-based eval/save — more appropriate than step-based for small data
    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,                  # Log every 25 steps (~every 0.13 epochs)

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LEARNING_RATE,       # FIX: 5e-5 instead of 2e-4

    #  FIX: Warmup ~1 epoch worth of steps
    warmup_steps=100,

    weight_decay=0.01,                 # Regularization to prevent overfitting

    num_train_epochs=EPOCHS,
    fp16=True,
    predict_with_generate=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,                # Keep only top 2 checkpoints
    report_to="none",
)

print("Training configuration:")
print(f"  Epochs:           {EPOCHS}")
print(f"  Batch size:       {BATCH_SIZE} x {GRAD_ACCUM_STEPS} grad accum = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"  Learning rate:    {LEARNING_RATE}")
print(f"  Warmup steps:     100")
print(f"  Early stopping:   patience=3 epochs")

Training configuration:
  Epochs:           20
  Batch size:       8 x 8 grad accum = 64 effective
  Learning rate:    5e-05
  Warmup steps:     100
  Early stopping:   patience=3 epochs


## 11. Train

In [12]:
#  NEW: EarlyStoppingCallback — stops training if eval_loss doesn't improve
# for 3 consecutive epochs. Prevents wasting GPU time on overfit epochs.
early_stopping = EarlyStoppingCallback(early_stopping_patience=3)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    processing_class=tokenizer,
    callbacks=[early_stopping],        #  NEW
)

print("Starting training...")
print("(Watch eval_loss: if it starts rising, early stopping will trigger)")
trainer.train()

# Save LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"\n LoRA adapter saved to {OUTPUT_DIR}/final")

Starting training...
(Watch eval_loss: if it starts rising, early stopping will trigger)


Epoch,Training Loss,Validation Loss
1,4.214900,3.731360
2,3.249900,2.879552
3,2.680700,2.536180
4,2.310700,2.372500
5,2.173300,2.286034
6,1.984700,2.187408
7,1.900600,2.144129
8,1.798500,2.134406
9,1.696100,2.097219
10,1.611900,2.083107



 LoRA adapter saved to ./dharwad-lora-model/final


## 12. (Optional) Merge LoRA Weights into Base Model
Merging creates a standalone model that doesn't require PEFT at inference time.
Useful for deployment — but requires more disk space.

In [13]:
#  NEW: Merge LoRA adapter into base weights for easier standalone inference
print("Merging LoRA weights into base model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/final_merged")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_merged")
print(f" Merged model saved to {OUTPUT_DIR}/final_merged")
print("   (This model can be loaded with AutoModelForSeq2SeqLM directly — no PEFT needed)")

Merging LoRA weights into base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


 Merged model saved to ./dharwad-lora-model/final_merged
   (This model can be loaded with AutoModelForSeq2SeqLM directly — no PEFT needed)


## 13. Inference — Load Fine-tuned Model

In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "ai4bharat/indictrans2-indic-indic-1B"
LORA_PATH = "./dharwad-lora-model/final"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model = model.to(device)
model.eval()

print(f" Fine-tuned model loaded on {device}")

Loading tokenizer...
Loading base model...
Loading LoRA adapter...
 Fine-tuned model loaded on cuda


## 14. Translation Function

In [ ]:
import torch

ip_infer = IndicProcessor(inference=True)  # inference=True during decoding

SRC_LANG = "kan_Knda"
TGT_LANG = "kan_Knda"

def translate_to_dharwad(bookish_text, num_beams=5, max_length=256):
    """
    Convert Bookish Kannada → Dharwad dialect using fine-tuned IndicTrans2.
    """
    # Wrap input in a list — IndicProcessor expects batch.
    # Source text DOES need the "TAG TAG <text>" prefix — _src_tokenize
    # parses and validates it explicitly.
    batch = ip_infer.preprocess_batch(
        [bookish_text],
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG
    )

    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        batch,
        padding="longest",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)


    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=num_beams,
            max_length=max_length,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
    result = ip_infer.postprocess_batch(decoded, lang=TGT_LANG)

    return result[0]
 
 
# Quick test
print("Testing translation function...")
test_out = translate_to_dharwad("ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ")
print(f"Input:    ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ")
print(f"Output:   {test_out}")
print(f"Expected: ನಾ ನಿನ್ನೇ ಶಾಲಿಗ್ ಹೋಗಿದ್ನಿ")

Testing translation function...
Input:    ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ
Output:   ನಾ ನಿನ್ನೆ ಶಾಲೀಗ ಹೋಗಿದ್ದೆ.
Expected: ನಾ ನಿನ್ನೇ ಶಾಲಿಗ್ ಹೋಗಿದ್ನಿ


## 15. Test the Fine-tuned Model

In [16]:
# Test sentences — mix of seen patterns and novel sentences
test_sentences = [
    # From training distribution
    "ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ",
    "ಅವನು ತನ್ನ ತಾಯಿಯೊಂದಿಗೆ ಮಾತನಾಡುತ್ತಿದ್ದಾನೆ",
    "ನಿಮಗೆ ಏನು ಬೇಕು?",
    # Novel sentences (not in training data)
    "ಅವಳು ನಿನ್ನೆ ಮಾರುಕಟ್ಟೆಗೆ ಹೋಗಿದ್ದಳು",
    "ನಾವು ಇಂದು ಮನೆಯಲ್ಲಿ ಉಳಿಯುತ್ತೇವೆ",
]

# Expected outputs based on dataset patterns
expected = [
    "ನಾನ್ ನಿನ್ನೇ ಶಾಲಿಗ್ ಹೋಗಿದ್ನಿ",
    "ಅವ ಅವಂದ ಅವ್ವ ಜೋಡಿ ಮಾತಾಡತಾನ್",
    "ನಿಮ್ಗ್ ಏನ್ ಬೇಕ್?",
    "(novel)",
    "(novel)",
]

print("=" * 65)
print("       BOOKISH KANNADA → DHARWAD DIALECT")
print("=" * 65)

for i, bookish in enumerate(test_sentences):
    dharwad = translate_to_dharwad(bookish)
    print(f"\n📖 Bookish:  {bookish}")
    print(f"🗣️  Dharwad:  {dharwad}")
    if expected[i] != "(novel)":
        print(f" Expected: {expected[i]}")
    else:
        print(f"   (Novel sentence — no expected output)")
    print("-" * 50)

       BOOKISH KANNADA → DHARWAD DIALECT

📖 Bookish:  ನಾನು ನಿನ್ನೆ ಶಾಲೆಗೆ ಹೋಗಿದ್ದೆ
🗣️  Dharwad:  ನಾ ನಿನ್ನೆ ಶಾಲೀಗ ಹೋಗಿದ್ದೆ.
 Expected: ನಾನ್ ನಿನ್ನೇ ಶಾಲಿಗ್ ಹೋಗಿದ್ನಿ
--------------------------------------------------

📖 Bookish:  ಅವನು ತನ್ನ ತಾಯಿಯೊಂದಿಗೆ ಮಾತನಾಡುತ್ತಿದ್ದಾನೆ
🗣️  Dharwad:  ಅವ ಅವಂದ ಅವ್ವ ಜೋಡಿ ಮಾತಾಡಾಕತ್ತಾನ.
 Expected: ಅವ ಅವಂದ ಅವ್ವ ಜೋಡಿ ಮಾತಾಡತಾನ್
--------------------------------------------------

📖 Bookish:  ನಿಮಗೆ ಏನು ಬೇಕು?
🗣️  Dharwad:  ನಿಮ್ಗ ಏನ ಬೇಕ?
 Expected: ನಿಮ್ಗ್ ಏನ್ ಬೇಕ್?
--------------------------------------------------

📖 Bookish:  ಅವಳು ನಿನ್ನೆ ಮಾರುಕಟ್ಟೆಗೆ ಹೋಗಿದ್ದಳು
🗣️  Dharwad:  ಅವ್ಳ ನಿನ್ನೆ ಮಾರ್ಕೆಟ್ಗ ಹೋಗಿದ್ದ್ಲ.
   (Novel sentence — no expected output)
--------------------------------------------------

📖 Bookish:  ನಾವು ಇಂದು ಮನೆಯಲ್ಲಿ ಉಳಿಯುತ್ತೇವೆ
🗣️  Dharwad:  ನಾವ ಇವತ್ತ ಮನ್ಯಾಗ ಉಳಿಯತೀವಿ
   (Novel sentence — no expected output)
--------------------------------------------------


## 16. Evaluation on Test Set

In [17]:
import json
import sacrebleu

# Load test set
test_data = []
with open("test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line.strip()))

print(f"Evaluating on {len(test_data)} test examples...")

references = []
hypotheses = []

for item in test_data:
    pred = translate_to_dharwad(item["src"])
    references.append(item["tgt"])
    hypotheses.append(pred)

# BLEU score
bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"\n{'=' * 40}")
print(f"BLEU Score: {bleu.score:.2f}")
print(f"{'=' * 40}")
print("\nNote: BLEU measures surface-level overlap.")
print("For dialect conversion, consider human evaluation as well.")

# Show a few examples
print("\nSample predictions:")
for i in range(min(5, len(test_data))):
    print(f"  Src: {test_data[i]['src']}")
    print(f"  Ref: {references[i]}")
    print(f"  Pred: {hypotheses[i]}")
    print()

Evaluating on 200 test examples...

BLEU Score: 21.12

Note: BLEU measures surface-level overlap.
For dialect conversion, consider human evaluation as well.

Sample predictions:
  Src: ಸಾಪ್ತಾಹಿಕ ಬ್ಯಾಡ್ಮಿಂಟನ್ ತರಗತಿಗಳು ನಾಳೆ ಪ್ರಾರಂಭವಾಗುತ್ತವೆ ಮತ್ತು ಮೇ ಒಂದಕ್ಕೆ ಕೊನೆಗೊಳ್ಳುತ್ತವೆ.
  Ref: ಸಾಪ್ತಾಹಿಕ ಬ್ಯಾಡ್ಮಿಂಟನ್ ತರಗತಿಗಳ ನಾಳೀಗ ಶುರು ಅಕ್ಕಾವ ಮತ್ತ ಮೇ ಒಂದಕ್ಕ ಮುಗಿತಾವ.
  Pred: ವಾರಕ್ಕೊಮ್ಮೆ ಬ್ಯಾಡ್ಮಿಂಟನ್ ತರಗತಿಗಳ ನಾಳೆ ಶುರು ಆಗುತ್ತವೆ ಮತ್ತ ಮೇ ಒಂದಕ್ಕ ಮುಗಿಯೋತ್ತಾವ.

  Src: ಚುನಾವಣಾ ಆಯೋಗದ ಪರವಾಗಿ ವಾದ ಮಂಡಿಸಿದ ವಕೀಲರು.
  Ref: ಚುನಾವಣಾ ಆಯೋಗದ ಪರ ವಾದ ಮಾಡಿದ ವಕೀಲ್ರ.
  Pred: ಚುನಾವಣಾ ಆಯೋಗದ ಬದಲ್ ವಾದ ಮಂಡಿಸಿದ ವಕೀಲ್ರ.

  Src: ನಾನು ಕಾರಲ್ಲಿ ಕರ್ಕೊಂಡು ಹೋಗುತ್ತೇನೆ ಅಂದರೆ ನನ್ನ ಸ್ನೇಹಿತರು ಎಲ್ಲಾ ಜಾಗ ಬಿಟ್ಟು ಆ ಮನಾಲಿಗೆ ಹೋಗಣ ಅನ್ನೋದಾ?
  Ref: ನಾ ಕಾರನ್ಯಾಗ ಕರ್ಕೊಂಡ ಹೋಗತೀನಿ ಅಂದ್ರ ನನ್ ದೋಸ್ತ್ರ ಎಲ್ಲಾ ಜಾಗ ಬಿಟ್ಟ ಆ ಮನಾಲಿಗ ಹೋಗುಣು ಅನ್ನೋದ?
  Pred: ನಾ ಕಾರ್ನ್ಯಾಗ ಕರ್ಕೊಂಡ ಹೋಗ್ತೀನಿ ಅಂದ್ರ ನನ್ ದೋಸ್ತ್ರ ಎಲ್ಲಾ ಜಾಗ ಬಿಟ್ಟ ಆ ಮನಾಲಿಗ ಹೋಗೋದ ಅನ್ನೋದ?

  Src: ಹೀಗಾಗಿ ಅವನು ಹಾಡುವುದನ್ನ ಬಿಟ್ಟು ಬೇರೆ ಯಾವುದಾದರೂ ಸಣ್ಣ ಕೆಲಸ ಮಾಡ್ಕೋಬಹುದು ಅಂತ ನನಗೆ  ತಳಮಳ ವಾಗುತ್ತಿದೆ.
  Ref: ಹಿಂಗಾಗಿ ಅವಾ ಹಾಡದ್‌ ಬಿ